# Data Preprocessing Pipeline

## 0) Setup
The file paths are relative to the project root, so the notebook first moves from `notebooks/` up to the project root.

In [1]:
import os

if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')

import pandas as pd
import joblib
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split

RAW_CSV_PATH = 'data/raw/weather_data.csv'
pd.set_option('display.precision', 3)

## 1) Load and inspect the raw data

In [2]:
df = pd.read_csv(RAW_CSV_PATH)
df.head()

,day_of_year,day_temp,humidity,cloud_cover,uv_index,day_type
0,150,32.0,75.6,0.0,5.9,Sunny
1,164,26.9,49.0,23.5,3.3,Sunny
2,188,33.2,43.6,34.7,4.2,Sunny
3,174,40.2,59.9,92.2,5.7,Cloudy
4,151,26.1,56.6,58.9,2.8,Cloudy


### Shape

In [3]:
print(f"Rows: {df.shape[0]}, Columns: {df.shape[1]}")

Rows: 1500, Columns: 6


### Data types

In [4]:
df.dtypes

day_of_year      int64
day_temp       float64
humidity       float64
cloud_cover    float64
uv_index       float64
day_type           str
dtype: object

### Summary statistics

In [5]:
df.describe()

,day_of_year,day_temp,humidity,cloud_cover,uv_index
count,1500.000,1500.000,1500.000,1500.000,1500.000
mean,171.975,28.381,60.161,44.268,4.219
std,69.033,7.874,19.101,24.274,2.407
min,1.000,5.000,10.000,0.000,0.000
25%,131.000,23.000,47.300,26.600,2.600
50%,182.500,28.400,59.950,44.450,4.100
75%,217.000,33.425,73.400,60.350,5.800
max,365.000,48.000,100.000,100.000,11.000


### Missing values per column

In [6]:
df.isna().sum()

day_of_year    0
day_temp       0
humidity       0
cloud_cover    0
uv_index       0
day_type       0
dtype: int64

## 2) Handle missing values

- **Features:** missing values are replaced with the column median, because the median is not influenced by outliers.
- **Target (`day_type`):** rows with no label are dropped, because a label cannot be guessed in preprocessing.

In [7]:
# Replace any NaN/None feature values with the column's median
for column in df.columns[:-1]:
    df[column] = df[column].fillna(df[column].median())
# Remove any unlabelled instances
df = df.dropna(subset=['day_type'])

print(f"Rows: {len(df)}, missing values remaining: {df.isna().sum().sum()}")

Rows: 1500, missing values remaining: 0


## 3) Remove duplicates and detect outliers
### Duplicates

In [8]:
print(f"Duplicate rows found: {df.duplicated().sum()}")
df = df.drop_duplicates()
print(f"Rows after removing duplicates: {len(df)}")

Duplicate rows found: 0
Rows after removing duplicates: 1500


### Outliers
A value is only treated as an error if it is physically impossible (e.g., humidity above 100%). Those rows are removed. Values considered unusual are those outside a specified range (specifically, outside 1.5 × IQR). Unusual values are counted, but not removed, because they cannot be assumed to be errors.

In [9]:
VALID_RANGES = {
    'day_of_year': (1, 366),
    'day_temp': (-60, 60),
    'humidity': (0, 100),
    'cloud_cover': (0, 100),
    'uv_index': (0, 20),
}

for column, (low, high) in VALID_RANGES.items():
    q1 = df[column].quantile(0.25)
    q3 = df[column].quantile(0.75)
    iqr = q3 - q1
    unusual = ((df[column] < q1 - 1.5 * iqr) | (df[column] > q3 + 1.5 * iqr)).sum()
    impossible = ((df[column] < low) | (df[column] > high)).sum()
    print(f"{column:12} unusual (IQR): {unusual:3}   impossible: {impossible}")

day_of_year  unusual (IQR):  22   impossible: 0
day_temp     unusual (IQR):   7   impossible: 0
humidity     unusual (IQR):   0   impossible: 0
cloud_cover  unusual (IQR):   0   impossible: 0
uv_index     unusual (IQR):   4   impossible: 0


In [10]:
# Remove only physically impossible values
rows_before = len(df)
for column, (low, high) in VALID_RANGES.items():
    df = df[df[column].between(low, high)]
print(f"Rows removed as outliers: {rows_before - len(df)}")

Rows removed as outliers: 0


The unusual are mostly very cold days and very high UV days. They days that help the model recognise the `Cold` and `Hot` classes, so they are kept.

## 4) Split into training, validation and test sets

70% training, 15% validation, 15% test. The sampling is stratified, keeping the same proportion of day types in every set.

In [11]:
train, temp = train_test_split(df, test_size=0.30, stratify=df['day_type'], random_state=42)
val, test = train_test_split(temp, test_size=0.50, stratify=temp['day_type'], random_state=42)
df_train, df_val, df_test = train.copy(), val.copy(), test.copy()

print(f"Train: {len(df_train)}, Validation: {len(df_val)}, Test: {len(df_test)}")

Train: 1050, Validation: 225, Test: 225


## 5) Scale the numeric features

A `MinMaxScaler` is fitted on the training set only (so that the scaler initially doesn't know of the other sets), then applied to all three sets.

In [12]:
numeric_cols = df.columns[:-1]
scaler = MinMaxScaler().fit(df_train[numeric_cols])
for split in (df_train, df_val, df_test):
    split[numeric_cols] = scaler.transform(split[numeric_cols])
df[numeric_cols] = scaler.transform(df[numeric_cols])
joblib.dump(scaler, "models/scaler.pkl")

df_train.describe().loc[['min', 'max']]

,day_of_year,day_temp,humidity,cloud_cover,uv_index
min,0.0,0.0,0.0,0.0,0.0
max,1.0,1.0,1.0,1.0,1.0


## 6) Save the processed data

In [13]:
df_train.to_csv('data/processed/weather_data_train.csv', index=False)
df_val.to_csv('data/processed/weather_data_val.csv', index=False)
df_test.to_csv('data/processed/weather_data_test.csv', index=False)